In [1]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy
from collections import defaultdict
import time
from multiprocessing import Pool, cpu_count
from functools import partial
from tqdm import tqdm
from joblib import Parallel, delayed

In [2]:
def sample_dense_core(df, target_ratings=100_000, core_threshold=5, random_state=42):
    """
    Performs Density-Biased Core Sampling.

    Args:
        df (pd.DataFrame): The full ratings dataframe with columns ['user_id', 'item_id', 'rating'].
        target_ratings (int): The desired approximate number of ratings in the final sample.
        core_threshold (int): The minimum number of interactions for users/items to be kept in the core.
        random_state (int): Random state for reproducibility.

    Returns:
        pd.DataFrame: A dense, interconnected sample of the original dataframe.
    """
    print("--- Starting Dense Core Sampling ---")
    
    # 1. Iterative Filtering to find the dense core
    print(f"Finding the dense core with threshold >={core_threshold}...")
    n_users_before, n_items_before = df['user_id'].nunique(), df['item_id'].nunique()
    
    while True:
        before_len = len(df)
        
        # Filter items
        item_counts = df['item_id'].value_counts()
        items_to_keep = item_counts[item_counts >= core_threshold].index
        df = df[df['item_id'].isin(items_to_keep)]
        
        # Filter users
        user_counts = df['user_id'].value_counts()
        users_to_keep = user_counts[user_counts >= core_threshold].index
        df = df[df['user_id'].isin(users_to_keep)]

        if len(df) == before_len:
            break
            
    n_users_after, n_items_after = df['user_id'].nunique(), df['item_id'].nunique()
    print(f"Dense core found: {len(df)} ratings, {n_users_after} users, {n_items_after} items.")
    print(f"(Removed {n_users_before - n_users_after} users and {n_items_before - n_items_after} items)")

    # 2. User-Centric Activity Sampling from the core
    print(f"\nSampling users from the core to reach ~{target_ratings} ratings...")
    
    # Get the final user counts from the dense core
    final_user_counts = df['user_id'].value_counts()
    
    # Sort users by activity to prioritize those with more data
    active_users_sorted = final_user_counts.index.tolist()
    
    # Shuffle for randomness, but we could also just take the top N most active
    # A little shuffle prevents always picking the same superusers if the data doesn't change
    np.random.seed(random_state)
    np.random.shuffle(active_users_sorted)
    
    selected_users = []
    accumulated_ratings = 0
    for user_id in active_users_sorted:
        if accumulated_ratings >= target_ratings:
            break
        
        selected_users.append(user_id)
        accumulated_ratings += final_user_counts[user_id]

    print(f"Selected {len(selected_users)} users, accumulating {accumulated_ratings} ratings.")
    
    # 3. Final DataFrame Creation
    # This is the crucial step: get ALL ratings for the selected users
    final_df = df[df['user_id'].isin(selected_users)].reset_index(drop=True)
    
    print("--- Dense Core Sampling Complete ---")
    return final_df

In [3]:
def sample_balanced_core(df, target_ratings=100_000, min_user_interactions=3, min_item_interactions=5, random_state=42):
    """
    Performs Balanced Threshold Sampling.

    This method is less aggressive than iterative filtering. It ensures every user and
    item in the resulting pool meets a minimum interaction threshold, preserving a
    better balance of users and item diversity.

    Args:
        df (pd.DataFrame): The full ratings dataframe.
        target_ratings (int): The desired approximate number of ratings.
        min_user_interactions (int): 'A' - Minimum ratings for a user to be considered.
        min_item_interactions (int): 'B' - Minimum ratings for an item to be considered.
        random_state (int): Seed for reproducibility.

    Returns:
        pd.DataFrame: A balanced, interconnected sample.
    """
    print("--- Starting Balanced Threshold Sampling ---")

    # 1. One-Pass Threshold Filtering (The Balanced Approach)
    print(f"Filtering with User Interactions >= {min_user_interactions} and Item Interactions >= {min_item_interactions}...")
    
    # First, filter by items to reduce the dataframe size
    item_counts = df['item_id'].value_counts()
    items_to_keep = item_counts[item_counts >= min_item_interactions].index
    df_filtered = df[df['item_id'].isin(items_to_keep)]
    
    # Then, filter by users on the already item-filtered data
    user_counts = df_filtered['user_id'].value_counts()
    users_to_keep = user_counts[user_counts >= min_user_interactions].index
    df_core = df_filtered[df_filtered['user_id'].isin(users_to_keep)]

    print(f"Filtered Core: {df_core.shape[0]} ratings, {df_core['user_id'].nunique()} users, {df_core['item_id'].nunique()} items.")

    # 2. User-Centric Sampling from the Balanced Core
    if df_core.shape[0] <= target_ratings:
        print("Filtered core is smaller than target size. Returning the entire core.")
        return df_core.reset_index(drop=True)

    print(f"\nSampling users from the core to reach ~{target_ratings} ratings...")
    core_user_counts = df_core['user_id'].value_counts()
    
    # Shuffle users for random selection
    np.random.seed(random_state)
    shuffled_users = core_user_counts.index.to_numpy()
    np.random.shuffle(shuffled_users)
    
    selected_users = []
    accumulated_ratings = 0
    for user_id in shuffled_users:
        if accumulated_ratings >= target_ratings:
            break
        selected_users.append(user_id)
        accumulated_ratings += core_user_counts[user_id]

    print(f"Selected {len(selected_users)} users, accumulating {accumulated_ratings} ratings.")

    # 3. Final DataFrame Creation
    final_df = df_core[df_core['user_id'].isin(selected_users)].reset_index(drop=True)
    
    print("--- Balanced Sampling Complete ---")
    return final_df

In [4]:
# # --- Your New Pipeline ---
# # Load the full dataset
# user_item_ratings_subratings = pd.read_csv('../data/filtered/hotelrec_filtered_00001.csv')
# full_ratings_df = user_item_ratings_subratings[['user_id', 'item_id', 'rating']]
# full_ratings_df = full_ratings_df.drop_duplicates(subset=['user_id', 'item_id'])

# print("Full dataset info:")
# print(f"Shape: {full_ratings_df.shape}")
# print(f"Users: {full_ratings_df['user_id'].nunique()}")
# print(f"Items: {full_ratings_df['item_id'].nunique()}")
# print("-" * 20)

# # Apply the smart sampling
# ratings_df = sample_balanced_core(full_ratings_df, target_ratings=1_000_000, min_user_interactions=3, min_item_interactions=3)

# print("\nFinal sampled dataset info:")
# print(f"Shape: {ratings_df.shape}")
# print(f"Users: {ratings_df['user_id'].nunique()}")
# print(f"Items: {ratings_df['item_id'].nunique()}")
# ratings_per_user = ratings_df.shape[0] / ratings_df['user_id'].nunique()
# print(f"Density (Avg Ratings/User): {ratings_per_user:.2f}")
# print("-" * 20)

# ratings_df['rating'] = ratings_df['rating'].astype(np.float16)
# ratings_df['user_id'] = ratings_df['user_id'].astype(np.int32)
# ratings_df['item_id'] = ratings_df['item_id'].astype(np.int32)

In [5]:
user_item_ratings_subratings = pd.read_csv('../data/filtered/dense_user_item_ratings.csv')
ratings_df = user_item_ratings_subratings[['user_id', 'item_id', 'rating']]
ratings_df = ratings_df.sample(n=250_000, random_state=42)
# drop duplicates
print(f"Original shape: {ratings_df.shape}")
# ratings_df = ratings_df.drop_duplicates(subset=['user_id', 'item_id'])
# print(f"Shape after dropping duplicates: {ratings_df.shape}")

# load data as half-precision floats and integers as half ints to save memory
ratings_df['rating'] = ratings_df['rating'].astype(np.float16)
ratings_df['user_id'] = ratings_df['user_id'].astype(np.int32)
ratings_df['item_id'] = ratings_df['item_id'].astype(np.int32)
ratings_df.info()

Original shape: (250000, 3)
<class 'pandas.core.frame.DataFrame'>
Index: 250000 entries, 952024 to 209809
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   user_id  250000 non-null  int32  
 1   item_id  250000 non-null  int32  
 2   rating   250000 non-null  float16
dtypes: float16(1), int32(2)
memory usage: 4.3 MB


In [6]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings_df[['user_id', 'item_id', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.05, random_state=42)

model = SVD(n_factors=50, n_epochs=100, lr_all=0.0125, reg_all=0.065, verbose=False)
model.fit(trainset)

In [7]:
# --- 5. Evaluate on the test set ---
print("\nEvaluating model performance...")
predictions = model.test(testset)

# Calculate RMSE (Root Mean Squared Error)
rmse = accuracy.rmse(predictions, verbose=False)
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

# Calculate MAE
mae = accuracy.mae(predictions, verbose=False)
print(f"Mean Absolute Error (MAE):      {mae:.4f}")



Evaluating model performance...
Root Mean Squared Error (RMSE): 0.8634
Mean Absolute Error (MAE):      0.6633


In [ ]:
# run optuna # --hyperparameter tuning-- to optimize model parameters for different n_factors, n_epochs, lr_all
import optuna

def objective(trial):
    n_factors = trial.suggest_int('n_factors', 20, 100)
    n_epochs = trial.suggest_int('n_epochs', 10, 100)
    lr_all = trial.suggest_float('lr_all', 0.001, 0.1, log=True)
    reg_all = trial.suggest_float('reg_all', 0.001, 0.1, log=True)

    model = SVD(n_factors=n_factors, n_epochs=n_epochs, lr_all=lr_all, reg_all=reg_all)
    model.fit(trainset)
    predictions = model.test(testset)
    rmse = accuracy.rmse(predictions, verbose=False)
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)  # Reduced number of trials for quicker execution
print("Best hyperparameters: ", study.best_params)


[I 2025-11-17 19:43:52,189] A new study created in memory with name: no-name-dcbcb28d-0e66-4ee0-9abd-9459b746d47f
[W 2025-11-17 19:43:56,665] Trial 0 failed with parameters: {'n_factors': 50, 'n_epochs': 54, 'lr_all': 0.009116670000463591, 'reg_all': 0.0061241201101136285} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/84/sbq4d0yx2vdcrx0_5_cy2hmh0000gn/T/ipykernel_7036/837636758.py", line 11, in objective
    model.fit(trainset)
  File "surprise/prediction_algorithms/matrix_factorization.pyx", line 155, in surprise.prediction_algorithms.matrix_factorization.SVD.fit
  File "surprise/prediction_algorithms/matrix_factorization.pyx", line 228, in surprise.prediction_algorithms.matrix_factorization.SVD.sgd
  File "/opt/anaconda3/lib/python3.11/site-packages/sur

KeyboardInterrupt: 

In [8]:
# --- (Worker function remains the same) ---
def worker_task_executor(task, model, trainset, all_items, k_list):
    user_inner, relevant_items = task
    watched_items = {item for (item, _) in trainset.ur[user_inner]}
    items_to_predict = ((user_inner, item_inner, 0) for item_inner in all_items 
                        if item_inner not in watched_items)

    predictions = model.test(items_to_predict, verbose=False)
    predictions.sort(key=lambda x: x.est, reverse=True)
    
    # This will store metrics for this user for each k
    user_results = {}
    
    # Evaluate for each k in the list
    for k in k_list:
        top_k_recs = {pred.iid for pred in predictions[:k]}
        hits = len(top_k_recs.intersection(relevant_items))
        
        precision = hits / k
        recall = hits / len(relevant_items)
        
        relevance_scores = [1 if item in relevant_items else 0 for item in [p.iid for p in predictions[:k]]]
        dcg = sum([rel / np.log2(i + 2) for i, rel in enumerate(relevance_scores)])
        ideal_k = min(k, len(relevant_items))
        idcg = sum([1 / np.log2(i + 2) for i in range(ideal_k)])
        ndcg = dcg / idcg if idcg > 0 else 0
        
        user_results[k] = {'precision': precision, 'recall': recall, 'ndcg': ndcg}
        
    return user_results

In [9]:
# --- MAIN EVALUATION FUNCTION USING JOBLIB ---
def evaluate_ranking_parallel_joblib(model, trainset, testset, k_list=[1,3,5,7,10], rating_threshold=4.0):
    ground_truth = defaultdict(list)
    for user_inner, item_inner, rating in testset:
        if rating >= rating_threshold:
            ground_truth[user_inner].append(item_inner)

    tasks = list(ground_truth.items())
    all_items = list(trainset.ir.keys())
    
    print(f"\nEvaluating for K={k_list} for {len(tasks)} users across {cpu_count()} CPU cores...")
    
    # Joblib call is the same, but passes the k_list to the worker
    all_user_results = Parallel(n_jobs=12, backend="loky")(
        delayed(worker_task_executor)(task, model, trainset, all_items, k_list)
        for task in tqdm(tasks)
    )

    # --- Aggregate results for each K ---
    final_metrics = []
    if not all_user_results:
        return pd.DataFrame()

    for k in k_list:
        avg_precision = np.mean([res[k]['precision'] for res in all_user_results])
        avg_recall = np.mean([res[k]['recall'] for res in all_user_results])
        avg_ndcg = np.mean([res[k]['ndcg'] for res in all_user_results])
        
        final_metrics.append({
            'K': k,
            'Precision': avg_precision,
            'Recall': avg_recall,
            'nDCG': avg_ndcg
        })
        
    return pd.DataFrame(final_metrics)

In [10]:
# 5. Evaluate the UNTUNED model
print("\n--- Evaluating the UNTUNED SVD model ---")
k_values = [1, 3, 5,7,10]
untuned_results_df = evaluate_ranking_parallel_joblib(model, trainset, testset, k_list=k_values)
print("\nResults for UNTUNED Model:")
print(untuned_results_df.round(4))


--- Evaluating the UNTUNED SVD model ---

Evaluating for K=[1, 3, 5, 7, 10] for 8460 users across 12 CPU cores...


  0%|          | 0/8460 [00:00<?, ?it/s]

 70%|██████▉   | 5916/8460 [09:33<04:13, 10.03it/s]/opt/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
100%|██████████| 8460/8460 [13:39<00:00, 10.33it/s]



Results for UNTUNED Model:
    K  Precision  Recall    nDCG
0   1     0.0001  0.0001  0.0001
1   3     0.0000  0.0001  0.0001
2   5     0.0000  0.0001  0.0001
3   7     0.0000  0.0002  0.0001
4  10     0.0000  0.0004  0.0002
